In [ ]:
# MongoClient łączy aplikację z serwerem MongoDB.
# `Faker` generuje realistyczne dane testowe, bez ręcznego wpisywania rekordów.
from pymongo import MongoClient  # pip install pymongo
from faker import Faker  # pip install faker

from utils.config import load_config

In [ ]:
config = load_config("config.yaml")
config

In [ ]:
# Poniższy kod tworzy listę rekordów z przykładowymi polami: imię, nazwisko, adres i e-mail.
# Takie dane pomagają sprawdzić, jak baza zachowuje się na rzeczywistych wartościach.
f = Faker("pl_PL")
dane = []
for _ in range(100):
    dane.append(
        {
            "imie": f.first_name(),
            "nazwisko": f.last_name(),
            "adres": f.address(),
            "email": f.email(),
            "data_urodzenia": f.date_of_birth(
                minimum_age=18, maximum_age=90
            ).isoformat(),
        }
    )

In [ ]:
dane

In [ ]:
# Połączenie do bazy odbywa się przez obiekt `MongoClient`.
# Następnie wybieramy bazę danych i kolekcję, w której zostaną zapisane dokumenty.
# Połącz z lokalną bazą MongoDB
client = MongoClient(
    host=config["mongo_db"]["host"],
    port=config["mongo_db"]["port"],
    username=config["mongo_db"]["user"],
    password=config["mongo_db"]["password"],
)

# Wybierz bazę danych
db = client[config["mongo_db"]["database"]]

# Wybierz kolekcję
collection = db[config["mongo_db"]["collection"]]


In [ ]:
client

In [ ]:
# `insert_one()` dodaje pojedynczy dokument do kolekcji.
# W praktyce pozwala wstawiać dane w małych porcjach lub w czasie rzeczywistym.
for record in dane:
    collection.insert_one(record)

In [ ]:
# `find()` zwraca wszystkie dokumenty z kolekcji, a `print()` pokazuje ich zawartość.
# To najprostszy sposób sprawdzenia, czy dane zostały poprawnie zapisane.
# Przykładowe pobranie dokumentów
for doc in collection.find():
    print(doc)

In [ ]:
# Wyszukiwanie po polu `nazwisko` pokazuje, jak działa filtrowanie dokumentów.
# W MongoDB jest to bardzo naturalna operacja i przypomina `WHERE` w SQL.
# wyszukiwanie dokumentów z nazwiskiem "Kowalski"
for doc in collection.find({"nazwisko": "Kowalski"}):
    print(doc)

In [ ]:
# Agregacja pozwala grupować dane i liczyć wystąpienia.
# W praktyce jest to bardzo przydatne do analizy statystyk i raportów.
# agregacja - zliczanie wystąpień imion
pipeline = [
    {"$group": {"_id": "$imie", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
]

wyniki = list(collection.aggregate(pipeline))
for wynik in wyniki:
    print(f"{wynik['_id']}: {wynik['count']}")